In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from imblearn.over_sampling import RandomOverSampler
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 读取数据
train_df = pd.read_csv('unsw-nb15/UNSW_NB15_training-set.csv')
test_df = pd.read_csv('unsw-nb15/UNSW_NB15_testing-set.csv')

# 移除无关列
train_df.drop(columns=['id', 'label'], inplace=True)
test_df.drop(columns=['id', 'label'], inplace=True)

# 检查是否有缺失值
print("Train missing values:", train_df.isnull().sum().sum())
print("Test missing values:", test_df.isnull().sum().sum())

# 需要进行 One-hot 编码的列
cols = ['proto', 'state', 'service']

# One-hot 编码函数
def one_hot_encode(df, cols):
    return pd.get_dummies(df, columns=cols, drop_first=False)

# 合并数据进行 One-hot 编码
combined_data = pd.concat([train_df, test_df])
combined_data = one_hot_encode(combined_data, cols)

# 分离特征和目标变量
target = combined_data.pop('attack_cat')

# 归一化函数
def normalize(df):
    scaler = StandardScaler()
    df[:] = scaler.fit_transform(df)
    return df

# 归一化数据
normalized_data = normalize(combined_data)

# 添加目标列
normalized_data["Class"] = target

# 分离特征和标签
X = normalized_data.drop(columns=['Class']).values
y = normalized_data['Class'].values
print(X.shape)
# 进行 One-hot 编码
encoder = OneHotEncoder(sparse=False)
y_onehot = encoder.fit_transform(y.reshape(-1, 1))

# 转换为 PyTorch 张量
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y_onehot, dtype=torch.float32)


Train missing values: 0
Test missing values: 0
(257673, 196)


E:\shendu\anaconda3\envs\pytorch1\lib\site-packages\sklearn\preprocessing\_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [18]:
import torch
import torch.nn as nn

class NSLKDDModel(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(NSLKDDModel, self).__init__()

        # TCN 模块配置
        self.tcn_block1 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=16, padding='same', dilation=1),
            nn.BatchNorm1d(32),
            nn.ReLU()
        )
        self.tcn_block2 = nn.Sequential(
            nn.Conv1d(32, 32, kernel_size=32, padding='same', dilation=2),
            nn.BatchNorm1d(32),
            nn.ReLU()
        )
        self.tcn_block3 = nn.Sequential(
            nn.Conv1d(32, 32, kernel_size=64, padding='same', dilation=4),
            nn.BatchNorm1d(32),
            nn.ReLU()
        )

        # 池化层
        self.pool = nn.AdaptiveMaxPool1d(input_dim // 4)

        # LSTM（双向）
        self.lstm = nn.LSTM(32, 64, num_layers=2, batch_first=True, bidirectional=True)

        # Transformer Encoder
        self.transformer_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=128, nhead=4, dim_feedforward=256, batch_first=True),
            num_layers=2
        )

        # 全连接层
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(input_dim // 4 * 128, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        # TCN 模块
        x = self.tcn_block1(x)  # 输出 (batch_size, 32, sequence_length)
        x = self.tcn_block2(x)  # 输出 (batch_size, 32, sequence_length)
        x = self.tcn_block3(x)  # 输出 (batch_size, 32, sequence_length)

        # 池化
        x = self.pool(x)  # 输出 (batch_size, 32, sequence_length / 4)

        # LSTM 输入需要 (batch_size, sequence_length, features)
        x = x.permute(0, 2, 1)  # 调整为 (batch_size, sequence_length / 4, 32)
        lstm_out, _ = self.lstm(x)  # 输出 (batch_size, sequence_length / 4, 128)

        # Transformer Encoder
        transformer_out = self.transformer_encoder(lstm_out)  # 输出 (batch_size, sequence_length / 4, 128)

        # Flatten and Dropout
        flatten = transformer_out.reshape(transformer_out.size(0), -1)  # 输出 (batch_size, sequence_length / 4 * 128)
        flatten = self.dropout(flatten)

        # 输出层
        outputs = self.fc(flatten)
        return outputs


In [19]:
from sklearn import metrics
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
# 设置超参数
batch_size = 64
epochs = 15
learning_rate = 0.0005
k=10
# 交叉验证
kfold = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
oversample = RandomOverSampler(sampling_strategy='minority')

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = nn.CrossEntropyLoss(reduction='none')(inputs, targets)
        pt = torch.exp(-ce_loss)  # 预测的概率
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

criterion = FocalLoss()
# 统计结果
oos_pred = []
# 初始化结果列表
oos_accuracies = []
last_fold_y_true = []
last_fold_y_pred = []

# 初始化模型
model = NSLKDDModel(input_dim=196, num_classes=10).to(device)
#criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 遍历折叠
for fold, (train_idx, test_idx) in enumerate(kfold.split(X, y), start=1):

    # 划分数据集
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # 过采样少数类
    X_train, y_train = oversample.fit_resample(X_train, y_train)

    # 使用 LabelEncoder 转换 y_train 和 y_test
    label_encoder = LabelEncoder()
    y_train = label_encoder.fit_transform(y_train)  # 转换成整数标签
    y_test = label_encoder.transform(y_test)

    # 确保 y_train 和 y_test 是 numpy 数组的 int 类型
    y_train = y_train.astype(np.int64)
    y_test = y_test.astype(np.int64)

    # 转换为 PyTorch 数据集
    train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
    test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

    # 加载数据
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        correct = 0
        total = 0

        # 使用 tqdm 显示进度条
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for batch_data, batch_labels in progress_bar:
            batch_data, batch_labels = batch_data.to(device), batch_labels.to(device)

            batch_data = batch_data.unsqueeze(1)  # 添加通道维度
            optimizer.zero_grad()
            outputs = model(batch_data)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer.step()

            # 累积损失和准确性
            epoch_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == batch_labels).sum().item()
            total += batch_labels.size(0)

            # 更新进度条
            progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

        # 计算每轮的平均损失和准确率
        epoch_loss /= len(train_loader)
        epoch_acc = correct / total
        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")    

    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch_data, batch_labels in test_loader:
            batch_data, batch_labels = batch_data.to(device), batch_labels.to(device)
            batch_data = batch_data.unsqueeze(1)  # 添加通道维度
            outputs = model(batch_data)
            _, preds = torch.max(outputs, 1)

            y_true.extend(batch_labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    # 测试集各类别数量
    # test_class_counts = Counter(y_true)
    # print("\nTest Set Class Distribution:")
    # for label, count in sorted(test_class_counts.items()):
    #     print(f"  Class {label}: {count}")

    # 计算准确率
    acc = metrics.accuracy_score(y_true, y_pred)
    oos_accuracies.append(acc)
    print(f"Fold {fold} Accuracy: {acc:.4f}")

    # 保存最后一折的结果
    if fold == kfold.get_n_splits():
        last_fold_y_true = y_true
        last_fold_y_pred = y_pred

# 保存模型
model_save_path = "modelU6.pth"
torch.save(model, model_save_path)
print(f"Complete model saved to {model_save_path}")

# 输出每一折的准确率
print("Fold Accuracies:")
for i, acc in enumerate(oos_accuracies, start=1):
    print(f"  Fold {i}: {acc:.4f}")
 

Epoch 1/15: 100%|██████████| 4929/4929 [01:12<00:00, 67.74it/s, loss=0.1844] 


Epoch 1/15 - Loss: 0.3169, Accuracy: 0.7988


Epoch 2/15: 100%|██████████| 4929/4929 [00:47<00:00, 103.67it/s, loss=0.1222]


Epoch 2/15 - Loss: 0.2361, Accuracy: 0.8312


Epoch 3/15: 100%|██████████| 4929/4929 [00:47<00:00, 103.15it/s, loss=0.2120]


...

Epoch 13/15 - Loss: 0.1305, Accuracy: 0.8858


Epoch 14/15: 100%|██████████| 4929/4929 [00:47<00:00, 103.83it/s, loss=0.1364]


Epoch 14/15 - Loss: 0.1301, Accuracy: 0.8868


Epoch 15/15: 100%|██████████| 4929/4929 [00:47<00:00, 103.31it/s, loss=0.1158]


Epoch 15/15 - Loss: 0.1296, Accuracy: 0.8873
Fold 10 Accuracy: 0.8425
Complete model saved to modelU6.pth
Fold Accuracies:
  Fold 1: 0.8038
  Fold 2: 0.8177
  Fold 3: 0.8242
  Fold 4: 0.8137
  Fold 5: 0.8289
  Fold 6: 0.8325
  Fold 7: 0.8303
  Fold 8: 0.8375
  Fold 9: 0.8120
  Fold 10: 0.8425
